# Deploy Model for Inference

In [ ]:
!pip install onnxruntime numpy pickle5

In [ ]:
import onnxruntime as rt
import numpy as np
import pickle

with open('../artifact/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

sess = rt.InferenceSession("../models/anomaly/1/model.onnx", providers=rt.get_available_providers())
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

print("Model loaded successfully!")
print(f"Input name: {input_name}")
print(f"Output name: {output_name}")

In [ ]:
def predict_anomaly(ip_parts, hour, is_night, is_weekend, action_code, has_resource):
    features = [ip_parts[0], ip_parts[1], ip_parts[2], ip_parts[3], 
                hour, is_night, is_weekend, action_code, has_resource]
    
    features_scaled = scaler.transform([features]).astype(np.float32)
    prediction = sess.run([output_name], {input_name: features_scaled})
    probability = np.squeeze(prediction)
    
    return probability, probability > 0.5

print("\nTest Case 1: Normal office login")
prob, is_anomaly = predict_anomaly([192, 168, 10, 15], 10, 0, 0, 0, 0)
print(f"  Probability: {prob:.4f}, Anomaly: {is_anomaly}")

print("\nTest Case 2: Late night download from external IP")
prob, is_anomaly = predict_anomaly([203, 0, 113, 77], 3, 1, 0, 3, 1)
print(f"  Probability: {prob:.4f}, Anomaly: {is_anomaly}")

print("\nTest Case 3: Failed login attempt")
prob, is_anomaly = predict_anomaly([10, 10, 10, 10], 15, 0, 0, 4, 0)
print(f"  Probability: {prob:.4f}, Anomaly: {is_anomaly}")